In [1]:
!pip install optuna

  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.11.0
    Uninstalling typing_extensions-4.11.0:
      Successfully uninstalled typing_extensions-4.11.0


In [10]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier
import optuna

In [4]:
# load data
X_train = pd.read_csv("train_transaction.csv")
X_val = pd.read_csv("val_transaction.csv")

X_val.head()

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,TransactionID,TransactionDT,TransactionAmt,card1,card2,card3,card5,...,R_emaildomain_yahoo.de,R_emaildomain_yahoo.es,R_emaildomain_yahoo.fr,R_emaildomain_ymail.com,R_emaildomain_nan,M4_M0,M4_M1,M4_M2,M4_nan,isFraud
0,43130,43130,43130,3030130,1035237,34.50,16163,551.0,150.0,226.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0
1,72303,72303,72303,3059303,1618570,226.00,15813,251.0,150.0,226.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0
2,293354,293354,293354,3280354,7243463,108.50,15497,490.0,150.0,226.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0
3,32957,32957,32957,3019957,831523,159.95,12544,321.0,150.0,226.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0
4,14777,14777,14777,3001777,414969,77.95,17188,321.0,150.0,226.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0


In [5]:
# Split target column from main data
y_train = X_train['isFraud']
X_train.drop(['isFraud'], axis=1, inplace=True)

y_val = X_val['isFraud']
X_val.drop(['isFraud'], axis=1, inplace=True)

In [6]:
# Clean up some unnecessary columns
X_train.head()
X_train.drop(['Unnamed: 0.2', 'Unnamed: 0.1', 'Unnamed: 0'], axis=1, inplace=True)
X_val.drop(['Unnamed: 0.2', 'Unnamed: 0.1', 'Unnamed: 0'], axis=1, inplace=True)

In [7]:
# Remove all email columns
X_train_v2 = X_train[X_train.columns.drop(list(X_train.filter(regex='email')))]
X_val_v2 = X_val[X_val.columns.drop(list(X_val.filter(regex='email')))]

In [8]:
def objective(trial):
    # set up ranges for hyperparams
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 1, 50)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 20)
    min_weight_fraction_leaf = trial.suggest_float('min_weight_fraction_leaf', 0.0, 0.5, step=0.1)
    max_samples = trial.suggest_float('max_samples', 0.5, 1.0, step=0.1)

    # define model
    classifier = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        min_weight_fraction_leaf=min_weight_fraction_leaf,
        max_samples=max_samples,
        random_state=42,
        n_jobs=-1,
        verbose=0,
    )

    # train, predict, and score
    # CHANGE: move to AUC-ROC returned instead of accuracy
    classifier.fit(X_train_v2, y_train)
    y_pred_proba = classifier.predict_proba(X_val_v2)[:, 1]
    auc_roc = roc_auc_score(y_val, y_pred_proba)
    return auc_roc

In [11]:
# run study to maximize score
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),  # reproducible search
)

# Change number of trials here (if applicable)
study.optimize(objective, n_trials=250)

[I 2026-08-17 18:40:54,277] A new study created in memory with name: no-name-4c1afa6d-bacf-4eeb-9003-bdd40c3c26e8
[I 2026-08-17 18:41:35,566] Trial 0 finished with value: 0.923107779679705 and parameters: {'n_estimators': 106, 'max_depth': 48, 'min_samples_split': 15, 'min_samples_leaf': 12, 'min_weight_fraction_leaf': 0.0, 'max_samples': 0.5}. Best is trial 0 with value: 0.923107779679705.
[I 2026-08-17 18:42:01,936] Trial 1 finished with value: 0.927816464852457 and parameters: {'n_estimators': 58, 'max_depth': 44, 'min_samples_split': 13, 'min_samples_leaf': 15, 'min_weight_fraction_leaf': 0.0, 'max_samples': 1.0}. Best is trial 1 with value: 0.927816464852457.
[I 2026-08-17 18:42:13,998] Trial 2 finished with value: 0.7974364225653507 and parameters: {'n_estimators': 175, 'max_depth': 11, 'min_samples_split': 5, 'min_samples_leaf': 4, 'min_weight_fraction_leaf': 0.1, 'max_samples': 0.8}. Best is trial 1 with value: 0.927816464852457.
[I 2026-08-17 18:42:21,596] Trial 3 finished wit

In [12]:
# print out best parameters and the best model's accuracy incase of a crash/enviroment shutdown
print(f"Best parameters: {study.best_params}")
print(f"Best accuracy: {study.best_value}")

Best parameters: {'n_estimators': 195, 'max_depth': 48, 'min_samples_split': 10, 'min_samples_leaf': 2, 'min_weight_fraction_leaf': 0.0, 'max_samples': 1.0}
Best accuracy: 0.9395205011036534
